In [1]:
# import tonic
# import torch

# to_frame = tonic.transforms.ToFrame(
#     sensor_size=tonic.datasets.NMNIST.sensor_size, time_window=1e3
# )
# test_dataset = tonic.datasets.NMNIST(".", transform=to_frame, train=False)

# # Define dataloader
# batch_size = 32

# data_loader = torch.utils.data.DataLoader(
#     test_dataset,
#     shuffle=True,
#     batch_size=batch_size,
#     collate_fn=tonic.collation.PadTensors(),
# )

In [2]:
# import numpy as np

# def salvar_dataset_npz(loader, arquivo):
#     dados = []
#     labels = []

#     i = 0
#     for batch in loader:

#         if i == 32:
#             break
    
#         i += 1

#         x, y = batch

#         if torch.is_tensor(x):
#             x = x.cpu().numpy()
#             y = y.cpu().numpy()

#         dados.append(x)
#         labels.append(y)

#     dados = [arr[:, :300] for arr in dados]
#     dados = np.concatenate(dados, axis=0)

#     labels = np.concatenate(labels, axis=0)

#     np.savez_compressed(
#         arquivo,
#         data=dados,
#         labels=labels
#     )

#     print(f"Arquivo '{arquivo}.npz' salvo.")

In [3]:
# salvar_dataset_npz(data_loader, "nir_examples/cnn_teste.npz")

# NeuroHls

In [4]:
from neuro_hls import *

In [5]:
neuro_hls = NeuroHls("z_test")

## Definindo a Implementação do Modelo

In [6]:
nir_file = "nir_examples/cnn_sinabs.nir"

In [7]:
model = neuro_hls.read_nir_file(nir_file)

In [8]:
print(model)

-------------------------------------------------------
Input ([ 2 34 34]) - layer name: 'input'
-------------------------------------------------------
	Is recurrent: NO
	Dependencies:

-------------------------------------------------------
Conv2d (input: [ 2 34 34], output: [16 16 16]) - layer name: '0'
-------------------------------------------------------
	Weight shape: (16, 2, 5, 5)
	Stride: [2 2], Padding: [1 1], Dilation: [1 1]
	Groups: 1, Bias shape: (16,)
	Is recurrent: NO
	Dependencies:
	   - input (ready)

-------------------------------------------------------
IF (input: [16 16 16], output: [16 16 16]) - layer name: '1'
-------------------------------------------------------
	Parameter shape: (16, 16, 16)
	r range: [1.0000, 1.0000]
	v_threshold range: [1.0000, 1.0000]
	v_reset range: [0.0000, 0.0000]
	Is recurrent: NO
	Dependencies:
	   - 0 (ready)

-------------------------------------------------------
Conv2d (input: [16 16 16], output: [16 16 16]) - layer name: '2'
---

**OBS:** Para debugar o modelo, é melhor usar float (mais rápido e sem erro de quantização).

In [9]:
neuro_hls.implement_model(model, use_float=True)

## Criando o Testbench

In [10]:
neuro_hls.define_test_dataset("nir_examples/cnn_teste.npz", data_is_binary=True, step_count=300, different_sample_per_step=True)

In [11]:
neuro_hls.create_testbench(total_samples=100, batch_size=2, debug_mode=True)

Total samples used: 100 of 1024
Batch size: 2
Total batches: 50
Testbench was created.


**OBS:** Se a simulação misteriosamente não rodar, considere diminuir o `batch_size` do `create_testbench`.

**OBS 2:** Para a rede convolucional, é preciso zerar os potenciais entre inferências. Por enquanto, fazer manualmente.

In [ ]:
neuro_hls.run_csim()